<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Tool Use & Function Calling
</b></font> </br></p>

---

In [ ]:
#@title  🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "M02-Tool-Use"
os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

In diesem Modul lernen Sie das zentrale Konzept agentischer Systeme kennen: **Tools**.

Tools sind Python-Funktionen, die ein KI-Agent gezielt aufrufen kann – für Aufgaben, die das Sprachmodell allein nicht lösen kann: aktuelle Daten abfragen, rechnen, externe Systeme ansprechen.

**Lernziel:** Sie können eigene Tools mit dem `@tool`-Decorator definieren, verstehen die automatische Schema-Generierung und können robuste Tools mit Fehlerbehandlung bauen.

**Themen dieses Moduls:**
- Was sind Tools und warum brauchen wir sie?
- Der `@tool`-Decorator: Typ-Hints, Docstrings, Schema-Generierung
- Erstes Tool bauen und direkt testen
- Fehlerbehandlung in Tools
- Tool-Traces in LangSmith analysieren

# 2 | Was sind Tools?
---

Ein **Tool** ist eine Python-Funktion, die ein Agent während seines TAO-Zyklus aufrufen kann.

| Eigenschaft | Beschreibung |
|---|---|
| **Name** | Eindeutige Bezeichnung (= Funktionsname) |
| **Beschreibung** | Docstring – erklärt dem LLM, *wann* das Tool passt |
| **Schema** | Automatisch aus Typ-Hints generiert (JSON Schema) |
| **Rückgabewert** | String oder serialisierbarer Python-Typ |

Das LLM sieht **nur das Schema** (Name + Beschreibung + Parameter-Typen), niemals den eigentlichen Code.

> 💡 **Merkregel:** Je klarer der Docstring, desto besser wählt der Agent das richtige Tool.

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  sequenceDiagram</font> </br></p>

diagram = '''
sequenceDiagram
    autonumber
    participant U as Benutzer
    participant A as Agent (LLM)
    participant T as Tool

    U->>A: "Wie warm ist 37°C in Fahrenheit?"
    A->>A: Prüft verfügbare Tool-Schemas
    A->>T: celsius_nach_fahrenheit(temperatur=37.0)
    T->>A: 98.6
    A->>U: "37 °C entsprechen 98,6 °F"
'''

mermaid(diagram, width=650)

# 3 | @tool Decorator
---

Der `@tool`-Decorator aus `langchain_core.tools` verwandelt eine normale Python-Funktion in ein LangChain-Tool.

**Was passiert dabei automatisch:**
1. **Name** wird aus dem Funktionsnamen abgeleitet
2. **Beschreibung** wird aus dem Docstring übernommen
3. **Schema** wird aus den Typ-Hints als JSON Schema generiert

**Pflichtangaben für gute Tools:**
- ✅ Typ-Hints für alle Parameter und den Rückgabewert
- ✅ Klarer Docstring: was macht das Tool, was gibt es zurück?
- ✅ Sprechende Parameternamen

Das generierte Schema ist exakt das, was das LLM sieht, wenn es entscheidet, ob und wie es das Tool aufruft.

In [ ]:
import json
from langchain_core.tools import tool

@tool
def celsius_nach_fahrenheit(temperatur: float) -> float:
    """Rechnet eine Temperatur von Celsius in Fahrenheit um. Gibt den Fahrenheit-Wert zurück."""
    return round(temperatur * 9 / 5 + 32, 2)

# Schema anzeigen – was der Agent "sieht"
mprint("## Tool-Schema (was das LLM sieht)")
mprint(f"**Name:** `{celsius_nach_fahrenheit.name}`")
mprint(f"**Beschreibung:** {celsius_nach_fahrenheit.description}")
schema_json = json.dumps(
    celsius_nach_fahrenheit.args_schema.model_json_schema(),
    indent=2,
    ensure_ascii=False
)
mprint(f"**Parameter-Schema:**\n```json\n{schema_json}\n```")

# 4 | Erstes Tool bauen
---



Wir bauen zwei praxisnahe Tools und testen sie direkt – ohne Agent.

**Tipp:** Tools immer zuerst isoliert testen, bevor sie in einen Agenten eingebunden werden. Das erleichtert das Debugging erheblich.

In [ ]:
from langchain_core.tools import tool

@tool
def ist_buerozeit(stunde: int) -> str:
    """Prüft ob eine Stunde des Tages (0–23) in die typische Bürozeit fällt (9–17 Uhr).
    Gibt 'Bürozeit' oder 'Außerhalb Bürozeit' zurück."""
    if 9 <= stunde <= 17:
        return "Bürozeit"
    return "Außerhalb Bürozeit"

# Alle verfügbaren Tools auflisten
tools = [celsius_nach_fahrenheit, ist_buerozeit]
mprint(f"**Definierte Tools:** `{[t.name for t in tools]}`")

In [ ]:
# .func() ruft die Python-Funktion direkt auf – kein Runnable, kein Tracing
mprint("## Tool-Tests")
mprint('---')
mprint(f"**37 °C → Fahrenheit:** {celsius_nach_fahrenheit.func(temperatur=37.0)} °F")
mprint(f"**100 °C → Fahrenheit:** {celsius_nach_fahrenheit.func(temperatur=100.0)} °F")
mprint(f"**0 °C → Fahrenheit:** {celsius_nach_fahrenheit.func(temperatur=0.0)} °F")
print()
mprint(f"**14 Uhr – Bürozeit?** {ist_buerozeit.func(stunde=14)}")
mprint(f"**20 Uhr – Bürozeit?** {ist_buerozeit.func(stunde=20)}")
mprint(f"**9 Uhr – Bürozeit?** {ist_buerozeit.func(stunde=9)}")

# 5 | Tool mit Fehlerbehandlung
---

Tools dürfen **keine ungefangenen Exceptions werfen** – das würde den Agent-Loop unterbrechen.

**Grundregeln für robuste Tools:**
- ✅ Eingaben validieren (Typen, Wertebereiche, erlaubte Werte)
- ✅ Fehler als informativen String zurückgeben, nicht als Exception
- ✅ Fehlermeldungen sollen dem LLM helfen, die Situation zu verstehen

Das LLM kann aus einer klaren Fehlermeldung ("Währung CHF nicht unterstützt, verfügbar: EUR, USD, GBP") selbst entscheiden, wie es weiter vorgeht.

In [ ]:
from langchain_core.tools import tool

@tool
def waehrung_umrechnen(betrag: float, von: str, nach: str) -> str:
    """Rechnet einen Geldbetrag zwischen EUR, USD und GBP um.
    Unterstützte Währungen: EUR, USD, GBP.
    Gibt das Ergebnis als formatierten String zurück."""
    kurse = {
        ("EUR", "USD"): 1.08, ("USD", "EUR"): 0.93,
        ("EUR", "GBP"): 0.85, ("GBP", "EUR"): 1.18,
        ("USD", "GBP"): 0.79, ("GBP", "USD"): 1.27,
    }
    von_norm = von.upper().strip()
    nach_norm = nach.upper().strip()

    if von_norm == nach_norm:
        return f"{betrag:.2f} {nach_norm} (keine Umrechnung nötig)"

    kurs = kurse.get((von_norm, nach_norm))
    if kurs is None:
        return (
            f"Fehler: Umrechnung von {von_norm} nach {nach_norm} nicht unterstützt. "
            f"Verfügbare Währungen: EUR, USD, GBP."
        )

    ergebnis = round(betrag * kurs, 2)
    return f"{betrag:.2f} {von_norm} = {ergebnis:.2f} {nach_norm} (Kurs: {kurs})"

In [ ]:
# .func() ruft die Python-Funktion direkt auf – kein Runnable, kein Tracing
mprint("## Fehlerbehandlung testen")
mprint(f"✅ Normal:              {waehrung_umrechnen.func(betrag=100.0, von='EUR', nach='USD')}")
mprint(f"✅ Gleiche Währung:     {waehrung_umrechnen.func(betrag=50.0, von='USD', nach='USD')}")
mprint(f"⚠️ Unbekannte Währung:  {waehrung_umrechnen.func(betrag=100.0, von='EUR', nach='CHF')}")

# 6 | LangSmith: Tool-Traces
---

LangSmith zeichnet automatisch alle Tool-Aufrufe auf.

**Was Sie in einem Tool-Trace sehen:**
- Eingabe-Parameter (exakt wie das LLM sie übergeben hat)
- Rückgabewert (was an den Agenten zurückgeht)
- Ausführungszeit in ms
- Eventuelle Fehler

Das Tracing ist bereits in der Setup-Cell konfiguriert (`LANGCHAIN_TRACING_V2`, `LANGCHAIN_ENDPOINT`, Projekt `M02-Tool-Use`).

Anschließend erscheint jeder `.invoke()`-Aufruf automatisch als Trace in [eu.smith.langchain.com](https://eu.smith.langchain.com).

In [ ]:
from langchain.chat_models import init_chat_model

# Tracing bereits in der Setup-Cell konfiguriert
config = {
    "run_name": "M02_Kap6_ToolTrace",
    "tags": ["M02", "tools", "langsmith"],
    "metadata": {"user_id": "student_01"}
}

# LLM initialisieren und Tools binden
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
llm_mit_tools = llm.bind_tools([celsius_nach_fahrenheit, ist_buerozeit, waehrung_umrechnen])

# Anfrage senden – erscheint automatisch als Trace in LangSmith
antwort = llm_mit_tools.invoke(
    "Wie viel Fahrenheit sind 20 Grad Celsius?",
    config=config
)

mprint("## LLM-Antwort mit Tool-Calls")
if antwort.tool_calls:
    for tc in antwort.tool_calls:
        mprint(f"- **Tool:** `{tc['name']}` | **Args:** `{tc['args']}`")
else:
    mprint(antwort.content)

# A | Aufgabe
---



Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.



<p><font color='black' size="5">
Eigene Tools für Ihren Arbeitskontext
</font></p>

Denken Sie an einen Prozess aus Ihrem Arbeitsumfeld und definieren Sie dafür mindestens **2 eigene Tools** mit dem `@tool`-Decorator.

**Anforderungen:**
- Typ-Hints für alle Parameter und den Rückgabewert
- Klarer Docstring (1–2 Sätze): was macht das Tool, was gibt es zurück?
- Fehlerbehandlung für ungültige Eingaben

**Optionale Teilaufgaben:**
- Bauen Sie ein drittes Tool, das eine externe Datenquelle simuliert (Dictionary als Mock-Datenbank)
- Testen Sie jedes Tool direkt mit `.invoke({...})`
- Prüfen Sie das generierte Schema mit `.args_schema.model_json_schema()`
- Aktivieren Sie LangSmith-Tracing und inspizieren Sie einen Tool-Trace